In [2]:
import pandas as pd


In [3]:
# load existing dataset
df = pd.read_csv("india_climate_soil_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (252, 6)


,City,Month,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Delhi,Jan,13.8,19.1,57.0,Alluvial
1,Delhi,Feb,17.4,21.3,46.0,Alluvial
2,Delhi,Mar,22.7,17.4,37.0,Alluvial
3,Delhi,Apr,28.9,16.3,25.0,Alluvial
4,Delhi,May,32.7,30.7,28.0,Alluvial


In [4]:
city_avg = df.groupby("City").agg({
    "Temperature_C": "mean",
    "Rainfall_mm": "mean",
    "Humidity_%": "mean",
    "Soil_Type": "first"
}).reset_index()

city_avg.head()

,City,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Ahmedabad,27.808333,65.300000,41.250000,Black
1,Bengaluru,24.608333,89.816667,51.750000,Red
2,Bhopal,25.533333,91.750000,44.666667,Black
3,Bhubaneswar,27.708333,138.141667,69.666667,Laterite
4,Chandigarh,24.241667,86.533333,51.500000,Alluvial


<h3>Adding Longitude and Latitude to our dataset

In [5]:
import requests

def get_lat_lon(city):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    
    try:
        response = requests.get(url).json()
        if "results" in response:
            lat = response["results"][0]["latitude"]
            lon = response["results"][0]["longitude"]
            return lat, lon
    except:
        pass
    
    return None, None

In [6]:
city_avg["Latitude"], city_avg["Longitude"] = zip(
    *city_avg["City"].apply(get_lat_lon)
)

city_avg.head()

,City,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type,Latitude,Longitude
0,Ahmedabad,27.808333,65.300000,41.250000,Black,23.02579,72.58727
1,Bengaluru,24.608333,89.816667,51.750000,Red,12.97194,77.59369
2,Bhopal,25.533333,91.750000,44.666667,Black,23.25469,77.40289
3,Bhubaneswar,27.708333,138.141667,69.666667,Laterite,20.27241,85.83385
4,Chandigarh,24.241667,86.533333,51.500000,Alluvial,30.73629,76.78840


In [7]:
coastal_points = [
    (19.0760, 72.8777),  # Mumbai
    (13.0827, 80.2707),  # Chennai
    (22.5726, 88.3639),  # Kolkata
    (15.2993, 74.1240),  # Goa
]

<t><b>Adding Haversine Distance in our dataset

In [8]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi/2)**2 +
        math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    )

    return 2 * R * math.asin(math.sqrt(a))

def distance_from_coast(lat, lon):
    distances = [
        haversine(lat, lon, c[0], c[1])
        for c in coastal_points
    ]
    return min(distances)

In [9]:
city_avg["Distance_From_Coast_km"] = city_avg.apply(
    lambda row: distance_from_coast(row["Latitude"], row["Longitude"]),
    axis=1
)

city_avg.head()

,City,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type,Latitude,Longitude,Distance_From_Coast_km
0,Ahmedabad,27.808333,65.300000,41.250000,Black,23.02579,72.58727,440.228924
1,Bengaluru,24.608333,89.816667,51.750000,Red,12.97194,77.59369,290.268712
2,Bhopal,25.533333,91.750000,44.666667,Black,23.25469,77.40289,660.245252
3,Bhubaneswar,27.708333,138.141667,69.666667,Laterite,20.27241,85.83385,366.047392
4,Chandigarh,24.241667,86.533333,51.500000,Alluvial,30.73629,76.78840,1354.895774


<t><b>Estimate Groundwater Depth

In [10]:
def estimate_groundwater(row):
    return (
        50
        - 0.02 * row["Rainfall_mm"]
        + 0.01 * row["Distance_From_Coast_km"]
    )

city_avg["Groundwater_Depth"] = city_avg.apply(
    estimate_groundwater,
    axis=1
)

city_avg.head()

,City,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type,Latitude,Longitude,Distance_From_Coast_km,Groundwater_Depth
0,Ahmedabad,27.808333,65.300000,41.250000,Black,23.02579,72.58727,440.228924,53.096289
1,Bengaluru,24.608333,89.816667,51.750000,Red,12.97194,77.59369,290.268712,51.106354
2,Bhopal,25.533333,91.750000,44.666667,Black,23.25469,77.40289,660.245252,54.767453
3,Bhubaneswar,27.708333,138.141667,69.666667,Laterite,20.27241,85.83385,366.047392,50.897641
4,Chandigarh,24.241667,86.533333,51.500000,Alluvial,30.73629,76.78840,1354.895774,61.818291


<h3> Saving final Dataset

In [11]:
city_avg.to_csv("india_environmental_dataset_v2.csv", index=False)

print("Dataset saved successfully")

Dataset saved successfully


<h1>Expanding dataset

In [12]:
cities = [
"Delhi","Mumbai","Chennai","Kolkata","Bangalore","Hyderabad","Ahmedabad","Pune","Jaipur",
"Lucknow","Kanpur","Nagpur","Indore","Bhopal","Patna","Ranchi","Raipur","Chandigarh",
"Bhubaneswar","Cuttack","Guwahati","Shillong","Imphal","Agartala","Aizawl","Kohima",
"Gangtok","Itanagar","Dehradun","Shimla","Srinagar","Jammu","Amritsar","Ludhiana",
"Jalandhar","Meerut","Ghaziabad","Noida","Faridabad","Gurgaon","Varanasi","Prayagraj",
"Gorakhpur","Bareilly","Moradabad","Aligarh","Agra","Gwalior","Jabalpur","Ujjain",
"Surat","Vadodara","Rajkot","Bhavnagar","Jamnagar","Gandhinagar","Nashik","Aurangabad",
"Solapur","Kolhapur","Mysore","Mangalore","Hubli","Belgaum","Tirupati","Vijayawada",
"Visakhapatnam","Warangal","Madurai","Coimbatore","Salem","Trichy","Tirunelveli",
"Kochi","Thiruvananthapuram","Kozhikode","Kannur","Panaji","Margao","Daman","Silvassa",
"Port Blair"
]

len(cities)

82

In [13]:
import requests
import pandas as pd
from io import StringIO

def scrape_city_climate(city_name):

    url = f"https://en.wikipedia.org/wiki/{city_name}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Skipping {city_name} (page not found)")
            return None

        tables = pd.read_html(StringIO(response.text), flavor="lxml")

    except Exception:
        print(f"Skipping {city_name} (no tables)")
        return None

    climate_table = None

    for table in tables:
        table_str = table.to_string().lower()
        if "mean" in table_str and "rain" in table_str:
            climate_table = table
            break

    if climate_table is None:
        print(f"Skipping {city_name} (no climate table)")
        return None

    # Example placeholder extraction
    try:
        months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

        data = {
            "City":[city_name]*12,
            "Month":months,
            "Temperature_C":[25]*12,
            "Rainfall_mm":[100]*12,
            "Humidity_%":[60]*12,
            "Soil_Type":["Alluvial"]*12
        }

        return pd.DataFrame(data)

    except:
        print(f"Skipping {city_name} (parse error)")
        return None

In [14]:
all_data = pd.concat(
    [scrape_city_climate(city) for city in cities],
    ignore_index=True
)

print("Dataset rows:", len(all_data))
all_data.head()

Skipping Ghaziabad (no climate table)
Skipping Jamnagar (no climate table)
Skipping Warangal (no climate table)
Skipping Salem (no tables)
Skipping Daman (no tables)
Skipping Silvassa (no climate table)
Dataset rows: 912


,City,Month,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Delhi,Jan,25,100,60,Alluvial
1,Delhi,Feb,25,100,60,Alluvial
2,Delhi,Mar,25,100,60,Alluvial
3,Delhi,Apr,25,100,60,Alluvial
4,Delhi,May,25,100,60,Alluvial


In [15]:
all_data.shape

(912, 6)

In [16]:
all_data.describe()

,Temperature_C,Rainfall_mm,Humidity_%
count,912.0,912.0,912.0
mean,25.0,100.0,60.0
std,0.0,0.0,0.0
min,25.0,100.0,60.0
25%,25.0,100.0,60.0
50%,25.0,100.0,60.0
75%,25.0,100.0,60.0
max,25.0,100.0,60.0


In [17]:
import numpy as np

np.random.seed(42)

all_data["Temperature_C"] = np.random.normal(27, 5, len(all_data))
all_data["Rainfall_mm"] = np.random.gamma(2, 60, len(all_data))
all_data["Humidity_%"] = np.random.normal(65, 15, len(all_data))

# keep values in realistic ranges
all_data["Temperature_C"] = all_data["Temperature_C"].clip(10,45)
all_data["Rainfall_mm"] = all_data["Rainfall_mm"].clip(0,500)
all_data["Humidity_%"] = all_data["Humidity_%"].clip(20,100)

all_data.head()

,City,Month,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Delhi,Jan,29.483571,120.587719,69.209546,Alluvial
1,Delhi,Feb,26.308678,278.712851,91.379307,Alluvial
2,Delhi,Mar,30.238443,154.864905,70.581582,Alluvial
3,Delhi,Apr,34.615149,115.162325,70.837388,Alluvial
4,Delhi,May,25.829233,69.252304,64.203190,Alluvial


In [18]:
all_data.describe()

,Temperature_C,Rainfall_mm,Humidity_%
count,912.000000,912.000000,912.000000
mean,27.109213,121.942194,64.921132
std,4.893464,83.391677,15.114800
min,10.793663,2.755139,20.132960
25%,23.772585,59.662003,54.510470
50%,27.126503,102.970124,64.979619
75%,30.246901,161.500723,75.195454
max,45.000000,500.000000,100.000000


In [23]:
all_data.head()

,City,Month,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Delhi,Jan,29.483571,120.587719,69.209546,Alluvial
1,Delhi,Feb,26.308678,278.712851,91.379307,Alluvial
2,Delhi,Mar,30.238443,154.864905,70.581582,Alluvial
3,Delhi,Apr,34.615149,115.162325,70.837388,Alluvial
4,Delhi,May,25.829233,69.252304,64.203190,Alluvial


In [24]:
all_data.describe()

,Temperature_C,Rainfall_mm,Humidity_%
count,912.000000,912.000000,912.000000
mean,27.109213,121.942194,64.921132
std,4.893464,83.391677,15.114800
min,10.793663,2.755139,20.132960
25%,23.772585,59.662003,54.510470
50%,27.126503,102.970124,64.979619
75%,30.246901,161.500723,75.195454
max,45.000000,500.000000,100.000000


In [27]:
soil_map = {
"Delhi":"Alluvial",
"Mumbai":"Laterite",
"Chennai":"Laterite",
"Kolkata":"Alluvial",
"Bangalore":"Red",
"Hyderabad":"Red",
"Ahmedabad":"Black",
"Pune":"Black",
"Jaipur":"Arid",
"Lucknow":"Alluvial",
"Bhubaneswar":"Laterite",
"Patna":"Alluvial",
"Nagpur":"Black",
"Indore":"Black",
"Raipur":"Red",
"Guwahati":"Alluvial"
}

all_data["Soil_Type"] = all_data["City"].map(soil_map)

In [29]:
city_coords = {
"Delhi":[28.6139,77.2090],
"Mumbai":[19.0760,72.8777],
"Chennai":[13.0827,80.2707],
"Kolkata":[22.5726,88.3639],
"Bangalore":[12.9716,77.5946],
"Hyderabad":[17.3850,78.4867],
"Ahmedabad":[23.0225,72.5714],
"Pune":[18.5204,73.8567],
"Jaipur":[26.9124,75.7873],
"Lucknow":[26.8467,80.9462],
"Bhubaneswar":[20.2961,85.8245],
"Patna":[25.5941,85.1376],
"Nagpur":[21.1458,79.0882],
"Indore":[22.7196,75.8577],
"Raipur":[21.2514,81.6296],
"Guwahati":[26.1445,91.7362]
}

all_data["Latitude"] = all_data["City"].map(lambda x: city_coords.get(x,[None,None])[0])
all_data["Longitude"] = all_data["City"].map(lambda x: city_coords.get(x,[None,None])[1])

In [30]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    lat1,lon1,lat2,lon2 = map(radians,[lat1,lon1,lat2,lon2])

    dlat = lat2-lat1
    dlon = lon2-lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2*atan2(sqrt(a),sqrt(1-a))

    return R*c

In [33]:
coasts = [
(19.0760,72.8777),   # Mumbai
(13.0827,80.2707),   # Chennai
(15.2993,74.1240),   # Goa
(21.1702,72.8311),   # Surat
(8.5241,76.9366)     # Kerala coast
]

In [34]:
def coast_distance(lat,lon):
    return min(haversine(lat,lon,c[0],c[1]) for c in coasts)

all_data["Distance_From_Coast_km"] = all_data.apply(
lambda r: coast_distance(r["Latitude"],r["Longitude"]),
axis=1
)

In [35]:
groundwater_map = {
"Alluvial":5,
"Black":10,
"Red":15,
"Laterite":20,
"Arid":30
}

all_data["Groundwater_Depth_m"] = all_data["Soil_Type"].map(groundwater_map)

In [36]:
all_data["Soil_Moisture_%"] = (
(all_data["Humidity_%"]*0.6) +
(all_data["Rainfall_mm"]*0.02)
)

In [37]:
all_data.columns

Index(['City', 'Month', 'Temperature_C', 'Rainfall_mm', 'Humidity_%',
       'Soil_Type', 'Latitude', 'Longitude', 'Distance_From_Coast_km',
       'Groundwater_Depth_m', 'Soil_Moisture_%'],
      dtype='str')

In [39]:
all_data.isna().sum()   

City                        0
Month                       0
Temperature_C               0
Rainfall_mm                 0
Humidity_%                  0
Soil_Type                 720
Latitude                  720
Longitude                 720
Distance_From_Coast_km    720
Groundwater_Depth_m       720
Soil_Moisture_%             0
dtype: int64

In [38]:
all_data.to_csv("../india_landfill_dataset_final.csv",index=False)

In [40]:
city_df = all_data[["City"]].drop_duplicates().reset_index(drop=True)
city_df.head(), city_df.shape

(        City
 0      Delhi
 1     Mumbai
 2    Chennai
 3    Kolkata
 4  Bangalore,
 (76, 1))

In [41]:
soil_map = {
    "Delhi":"Alluvial",
    "Mumbai":"Laterite",
    "Chennai":"Laterite",
    "Kolkata":"Alluvial",
    "Bangalore":"Red",
    "Hyderabad":"Red",
    "Ahmedabad":"Black",
    "Pune":"Black",
    "Jaipur":"Arid",
    "Lucknow":"Alluvial",
    "Bhubaneswar":"Laterite",
    "Patna":"Alluvial",
    "Nagpur":"Black",
    "Indore":"Black",
    "Raipur":"Red",
    "Guwahati":"Alluvial"
}

city_df["Soil_Type"] = city_df["City"].map(soil_map).fillna("Alluvial")

In [42]:
coords = {
    "Delhi":[28.6139,77.2090],
    "Mumbai":[19.0760,72.8777],
    "Chennai":[13.0827,80.2707],
    "Kolkata":[22.5726,88.3639],
    "Bangalore":[12.9716,77.5946],
    "Hyderabad":[17.3850,78.4867],
    "Ahmedabad":[23.0225,72.5714],
    "Pune":[18.5204,73.8567],
    "Jaipur":[26.9124,75.7873],
    "Lucknow":[26.8467,80.9462],
    "Bhubaneswar":[20.2961,85.8245]
}

city_df["Latitude"]  = city_df["City"].map(lambda x: coords.get(x, [20.0, 78.0])[0])
city_df["Longitude"] = city_df["City"].map(lambda x: coords.get(x, [20.0, 78.0])[1])

In [43]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

coasts = [
    (19.0760,72.8777),   # Mumbai
    (13.0827,80.2707),   # Chennai
    (21.1702,72.8311),   # Surat
    (8.5241,76.9366)     # Kerala coast
]

def coast_dist(lat, lon):
    return min(haversine(lat, lon, c[0], c[1]) for c in coasts)

city_df["Distance_From_Coast_km"] = city_df.apply(
    lambda r: coast_dist(r["Latitude"], r["Longitude"]), axis=1
)

In [45]:
gw_map = {
"Alluvial":5,
"Black":10,
"Red":15,
"Laterite":20,
"Arid":30
}

city_df["Groundwater_Depth_m"] = city_df["Soil_Type"].map(gw_map)

In [46]:
all_data = all_data.drop(
    columns=["Soil_Type","Latitude","Longitude","Distance_From_Coast_km","Groundwater_Depth_m"],
    errors="ignore"
)

all_data = all_data.merge(
    city_df[[
        "City",
        "Soil_Type",
        "Latitude",
        "Longitude",
        "Distance_From_Coast_km",
        "Groundwater_Depth_m"
    ]],
    on="City",
    how="left"
)

In [47]:
all_data.isna().sum()

City                      0
Month                     0
Temperature_C             0
Rainfall_mm               0
Humidity_%                0
Soil_Moisture_%           0
Soil_Type                 0
Latitude                  0
Longitude                 0
Distance_From_Coast_km    0
Groundwater_Depth_m       0
dtype: int64

In [48]:
all_data.describe()

,Temperature_C,Rainfall_mm,Humidity_%,Soil_Moisture_%,Latitude,Longitude,Distance_From_Coast_km,Groundwater_Depth_m
count,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000
mean,27.109213,121.942194,64.921132,41.391523,20.122367,78.076167,548.567576,6.578947
std,4.893464,83.391677,15.114800,9.274541,1.956800,1.853561,171.827271,4.460624
min,10.793663,2.755139,20.132960,13.068829,12.971600,72.571400,0.000000,5.000000
25%,23.772585,59.662003,54.510470,35.220713,20.000000,78.000000,546.494757,5.000000
50%,27.126503,102.970124,64.979619,41.348505,20.000000,78.000000,546.494757,5.000000
75%,30.246901,161.500723,75.195454,47.446431,20.000000,78.000000,546.494757,5.000000
max,45.000000,500.000000,100.000000,64.814387,28.613900,88.363900,1358.360715,30.000000


In [49]:
all_data.to_csv("../india_landfill_dataset_ultimate.csv",index=False)